# À FAIRE 

## Demande 02/10/2026

J'ai commencé à travailler sur les anomalies (dans des tableaux pas très simples à utiliser il faut bien le dire...)

Mes remarques générales :

1er tableau
- onglet loc ko : 
les périodiques des Archives localisés chez nous devraient être en "pério pat" non ? est-ce que ça peut être fait de manière automatique ?
les périodiques pour les Ehpad sont rattachés au zèbre (décision de Gaëlle et Adeline). Ceux qui sont en transfert ou prêtés sont sur le site méd et je ne sais pas modifier (Lucas ?)

- onglet prix : 
l'information sur le prix de chaque fascicule de pério n'est pas rempli et ce n'est pas une erreur

- onglet code coll ko  : 
je ne comprends pas l'erreur sur les périodiques : ils sont en coll P17 ou en pério pat : Lucas j'ai besoin de ton aide pour comprendre

2e tableau 
Concernant les livres en magasin (sauf BD) qu'ils soient en retrait, en traitement ou en réparation, je vérifierai en magasin pour tous, sauf si vous me dîtes que vous en avez dans vos bureaux. Pouvez-vous faire un tour de vos bureaux pendant les vacances ?

- onglet doc en retrait : 
figurent dans cette liste les livres précieux : je n'irai évidement pas vérifier dans quels cartons ils se trouvent donc je les laisse en retrait.

- onglet doc en traitement : 
j'ai passé les livres précieux de ce statut au statut en retrait : comme on ne sait pas faire la différence entre les livres précieux en retrait de ceux en traitement, au moins ils ont tous le même statut.

- onglet doc en réparation
idem pour les livres précieux passés à en retrait

- onglet en reliure 
idem pour les livres précieux passés à en retrait  


* [x] Ajouter la côte au listing

## Demande 2
collection erronée : du style un doc avec une cote magasin ou reserve qui a un code collection "périodiques patrimoniaux" ou "BD".. 
localisation : mag RES 8 et qui a une cote qui commencent par autre chose que RES 8/...

In [1]:
import pandas as pd 
from datetime import datetime as dt
from kiblib.utils.db import DbConn

In [2]:
db_conn = DbConn().create_engine()

In [42]:



class AnomaliesExemplaires:
    
    def __init__(self,deadline=None,deadline_unit=None):
        
        query = """SELECT 
            i.dateaccessioned,
            i.barcode,
            i.homebranch,
            i.location,
            location.lib AS 'location_lib',
            i.ccode,
            ccode.lib AS 'ccode_lib',
            i.itype,
            b.title,
            i.itemcallnumber,
            i.onloan,
            i.price,
            i.replacementprice,
            i.notforloan,
            notforlan.lib AS 'statut',
            i.itemlost,
            itemlost.lib AS 'perdu',
            i.itemlost_on,
            i.damaged,
            itemdamaged.lib AS 'abime',
            i.damaged_on,
            i.withdrawn,
            withdrawn.lib AS 'retire_circulation',
            i.withdrawn_on,
            i.itemnotes,
            i.itemnotes_nonpublic,
            datelastseen
        FROM koha_prod.items i
        LEFT JOIN koha_prod.biblio b ON b.biblionumber = i.biblionumber
        LEFT JOIN koha_prod.authorised_values location ON location.authorised_value = i.location 
        LEFT JOIN koha_prod.authorised_values ccode ON ccode.authorised_value = i.ccode 
        LEFT JOIN koha_prod.authorised_values itemdamaged ON itemdamaged.authorised_value = i.damaged 
        LEFT JOIN koha_prod.authorised_values itemlost ON itemlost.authorised_value = i.itemlost 
        LEFT JOIN koha_prod.authorised_values withdrawn ON withdrawn.authorised_value = i.withdrawn
        LEFT JOIN koha_prod.authorised_values notforlan ON notforlan.authorised_value = i.notforloan 
        WHERE location.category = 'LOC' 
        AND ccode.category = 'COLLECTION'
        AND itemdamaged.category = 'DAMAGED' 
        AND itemlost.category = 'LOST'
        AND notforlan.category = 'Etat'
        AND withdrawn.category = 'RETIRECOLL'"""
        
        self.items = pd.read_sql(query,db_conn)
       
       # Exclusion des notices du Musée
       
        
        columns2datetime = ["dateaccessioned",'itemlost_on','damaged_on','withdrawn_on','onloan']
       
        for column in columns2datetime:
            pd.to_datetime(self.items[column])
       
        
       
       
    def get_DonneesIncoherentes(self):
        return("...")

    
    def get_DonneesManquantes(self):

        #Code-barres manquant
        self.items.loc[(self.items['barcode'].isna()) &
                       (~self.items['notforloan'].isin([-2,-1])),
                       ["anomalie - champ vide - code-barre"]] = 1

        # Localisation manquante
        self.items.loc[(self.items['location'].isna()) &
                       (~self.items['notforloan'].isin([-1,-2,-4])),
                      ['anomalie - champ vide - localisation']] = 1

        # Prix & Prix de remplacement manquant
        
        self.items.loc((~self.items["location"].isin(["CART","MED3B","MED3C","MED3D","MED3E","MED3F","MED3I","MED3I","MED3I"]))&
                       (~self.items["itype"]!="PRETPER") &
                       (~self.items["notforloan"].isin([-1,-2])) &
                       (self.items["price"].isna())&
                       (self.items["replacementprice"].isna())
                      )


        self.items.loc[(self.items['ccode'].isna()) &
                       (~self.items['notforloan'].isin([-1,-2,4])),
                       ["anomalie - champ vide - code collection"]] = 1

        self.items.loc[(self.items['itemcallnumber'].isna()) & 
                       (~self.items['notforloan'].isin([-1,-2,-4])),
                       ["anomalie - champ vide - cote"]]= 1

        self.items.loc[(self.items['itype'].isna()) & 
                       (~self.items['notforloan'].isin([-1,-2,-4])),
                       ["anomalie - champ vide - type document"]] = 1
        
        
    def get_ProblemesStatus(self):
        
        # Définition des deadlines
       
        deadline_3_mois = pd.to_datetime("today") - pd.Timedelta(3*31,unit='D')
        deadline_6_mois = pd.to_datetime("today") - pd.Timedelta(365/2,unit='D')
        deadline_1_an = pd.to_datetime("today") - pd.Timedelta(365,unit='D')
        deadline_2_ans = pd.to_datetime("today") - pd.Timedelta(365*2,unit='D')
       
        # Pour les perdus
        deadline_1_semaine = pd.to_datetime("today") - pd.Timedelta(7,unit='D')
        deadline_3_semaines = pd.to_datetime("today") - pd.Timedelta(7*3,unit='D')
        deadline_5_semaines = pd.to_datetime("today") - pd.Timedelta(7*5,unit='D')
        
        self.items.loc[(self.items['notforloan'].isin([-3])) &
                       (self.items['datelastseen']<deadline_1_an),
                      'anomalie - pb statut - En retrait > 1 an'] = 1
        
        self.items.loc[(self.items['notforloan'].isin([-1])) &
                       (self.items['datelastseen']<deadline_1_an),
                      'anomalie - pb statut - En commande > 1 an'] = 1
        
        self.items.loc[(self.items['notforloan'].isin([-2])) &
                       (self.items['datelastseen']<deadline_1_an),
                      'anomalie - pb statut - En traitement > 1 an'] = 1
        
        self.items.loc[(self.items['notforloan'].isin([-4])) &
                       (self.items['datelastseen']<deadline_1_an),
                      'anomalie - pb statut - En réparation > 1 an'] = 1
        
        self.items.loc[(self.items['notforloan'].isin([-3])) &
                       (self.items['datelastseen']<deadline_1_an),
                      'anomalie - pb statut - En retrait > 1 an'] = 1
        
        self.items.loc[(self.items['notforloan'].isin([5])) &
                       (self.items['datelastseen']<deadline_1_an),
                      'anomalie - pb statut - En reliure > 1 an'] = 1
        
        #self.cols2keep = ['barcode','homebranch','location_lib','title','ccode_lib','itemcallnumber']
        self.cols2keep_anomalies = []
        for column in items.items.columns:
            if 'anomalie' in column:
                self.cols2keep_anomalies.append(column)
                #self.cols2keep.append(column)

        self.items['total anomalies exemplaires'] = self.items[self.cols2keep_anomalies].sum(axis=1)

        self.anomalies = self.items[self.items['total anomalies exemplaires']>0]
        
    def save_AnomaliesExemplaires(self):
        
        self.cols2keep = ['barcode','homebranch','location_lib','title','ccode_lib','itemcallnumber']
        for i in self.cols2keep_anomalies:
            self.cols2keep.append(i)
        timestamp = dt.today().strftime("%Y%m%d")
        filepath = "/home/kibini/kibini2/data/collections/anomalies/"
        self.anomalies[self.cols2keep].to_excel(f"{filepath}{timestamp}_anomalies_exemplaires.xlsx")

    

In [43]:
items = AnomaliesExemplaires()

In [44]:
items.get_DonneesManquantes()
items.get_ProblemesStatus()
items.save_AnomaliesExemplaires()

TypeError: bad operand type for unary ~: 'str'

In [ ]:
items.anomalies[(items.anomalies["location_lib"].str.contains("Mag"))&
                            (items.anomalies["price"].isna()) &
                            (items.anomalies["replacementprice"].isna())
               ][["barcode","location","location_lib","itype","price","replacementprice"]]

In [6]:
#items.anomalies[items.cols2keep].to_excel("/home/kibini/kibini2/data/collections/anomalies/prout.xlsx")

In [7]:
"""
# Anomalies (prix vide exclus) avec plusieurs anomalies (seulement 6 au 30 janvier 2026)
items.items[(items.items['total anomalies exemplaires']>1) &
            (items.items['anomalie - champ vide - prix'].isna())
           ].to_excel('/home/kibini/kibini2/data/collections/anomalies/test_anomalies_multipltes.xlsx')
"""

"\n# Anomalies (prix vide exclus) avec plusieurs anomalies (seulement 6 au 30 janvier 2026)\nitems.items[(items.items['total anomalies exemplaires']>1) &\n            (items.items['anomalie - champ vide - prix'].isna())\n           ].to_excel('/home/kibini/kibini2/data/collections/anomalies/test_anomalies_multipltes.xlsx')\n"